In [3]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

True

# LLM

In [4]:
import httpx
llm = ChatGroq(model="openai/gpt-oss-20b",
                      http_client=httpx.Client(verify=False))

# Transcript

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi

In [24]:
url='https://youtu.be/nlz9j-r0U9U?si=mQpsKgOwCtdMvEED'
video_id='EpjLT1qC44g'
context = YouTubeTranscriptApi().fetch(video_id, languages=['hi'])

In [25]:
transcript = " ".join(chunk.text for chunk in context)

# Splitter

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [27]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

In [28]:
text = splitter.split_text(transcript)
len(text)

247

# Indexing (Embeddings and Vector DB)

In [29]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [30]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [31]:
vs = Chroma.from_texts(text, embeddings)

In [32]:
retriever = vs.as_retriever(searchtype="similarity", search_kwargs={'k':5})

In [33]:
retriever.invoke("what are the laws of motion?")

[Document(id='43a42126-1111-4d43-b33b-4ffbade83a6a', metadata={}, page_content='खत्म पावर पावर क्या शक्ति अद्भुत धम्य साहस की परिभाषा है यह मानवता की आशा है होता है जब आदमी को अपना ज्ञान कहलाया वो शक्तिमान पावर मान ठीक है एक पावर मैन भी आता था तो पावर का मतलब बेसिकली यह है द रेट ऑफ डूइंग वर्क द रेट ऑफ डूइंग वर्क इज डिफाइंड एस पावर इसका मतलब क्या समझते एक आदमी को मैंने बोला जा दौड़ लगा के 200 किलोमीटर व 100 मीटर एक को बोला जा तू भी लगा के 100 मीटर जो जल्दी मेरे पास वापस आया मतलब उसकी ज्यादा पावर थी कितने टाइम में कितना काम किया इसको ही बोलते है पावर ट्स इट एसआई यूनिट ऑफ पावर इज एसआई यूनिट इज वट डब्लू ए डटी या फिर पावर का फर्मूला देख लो पावर का फार्मूला वर्क अपॉन टाइम ये होता है तो जूल पर सेकंड ये भी है ठीक है ये कैसी क्वांटिटी है स्केलर क्वांटिटी स्केलर क्वांटिटी तो ये कुछ कुछ बातें जो तुम्हे पता होनी चाहिए पावर के बहुत सारे फार्मूले निकलते चलो दिखाए अब मैं बोलू'),
 Document(id='6c30b8b9-8c0a-408b-87c1-9227297ef78e', metadata={}, page_content='pt01 पावर का एक फार्मूला होगा क्या डेल्टा ऑ

In [34]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""You are an experienced educator\n 
    Your goal is to answer the user query based on the given context only, strictly no hallucination, if you can't answer something from the context then simply say 'I don't know'
    Even if the context is in hindi, translate it into english and answer the query in english only.
    
    context: {context},
    quert: {query}
    """,
    input_variables=['context', 'query']
)

In [35]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [36]:
def format_context(retriver_docs):
    result = "\n\n".join(doc.page_content for doc in retriver_docs)
    return result

In [37]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

In [38]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_context),
    'query' : RunnablePassthrough()
})

In [39]:
chain = parallel_chain | prompt | llm | parser

In [42]:
resp = chain.invoke("create the short notes of this video?")
print(resp)

**Short Notes (English Translation of the Video)**  

1. **Self‑Numerical Practice**  
   - The teacher stresses doing as many numerical problems as possible.  
   - Working through the problems yourself (thinking about each step) builds confidence, unlike simply looking at the solution.  
   - Start with a slow, deliberate approach; later, when you’re comfortable, you can speed up.

2. **Percentage Calculations**  
   - Remember the basic rule:  
     *Percent change = (new value – original value) / original value × 100.*  
   - To find the remaining percentage after a subtraction, subtract the percentage used from 100.  
   - Even if you haven’t formally learned percentages, this rule is fundamental and will help you with any increase/decrease problems.

3. **Kinetic Energy (KE) and Momentum**  
   - KE is proportional to the square of velocity:  
     \[
     KE = \frac{1}{2}mv^2
     \]  
   - Because velocity is proportional to momentum (p = mv), an increase in momentum results in